In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import sigpy as sp
import sigpy.plot as pl
import numpy as np
import os
import sys
from pathlib import Path
import jax as jx
import time

#ksp = np.load(r"/Users/ayman/Library/CloudStorage/OneDrive-King'sCollegeLondon/Project/Sigpytutorial/projection_ksp.npy")
#coord = np.load(r"/Users/ayman/Library/CloudStorage/OneDrive-King'sCollegeLondon/Project/Sigpytutorial/projection_coord.npy")

sys.path.insert(0,'/Users/ayman/Desktop/MSc Project Local/MSc-Project-ZTE')
import aymansigmri as asm


sys.path.insert(0, "/Users/ayman/Documents/GitHub/riesling/python")

import riesling as rlp



#riesling_bin = Path.home() / "Documents" / "github" / "riesling" / "build" / "cxx" / "riesling"
#os.environ["PATH"] += os.pathsep + str(riesling_bin)

In [2]:
lowres = '/Users/ayman/Desktop/MSc Project Local/MSc-Project-ZTE/riselingwork/lores'
coorlowres = rlp.io.read_trajectory(f'{lowres}.h5')
coorlowres = np.asarray(coorlowres)
kspacelowres = rlp.io.read_data(f'{lowres}.h5')
kspacelowres = np.asarray(kspacelowres)
kspacelowres = kspacelowres[0, 0] 
kspacelowres = np.moveaxis(kspacelowres, -1, 0)



In [3]:
dcf_new = np.sqrt(coorlowres[...,0]**2 + coorlowres[...,1]**2 + coorlowres[...,2]**2)

zte_im_grid = sp.nufft_adjoint(kspacelowres* dcf_new, coorlowres)
gridded_data = sp.fft(zte_im_grid, axes=(-2, -1))
#zte_im = np.sum(np.abs(zte_im_grid)**2, axis=0)**0.5

In [4]:
width=4
gap = 2
num_iters = 50

resize_wid=160
rw_div2 = int(resize_wid/2)

inner_wid=30
indiv2 = int(inner_wid/2)

In [6]:
enlarged_cartesian = asm.zero_padding3d(gridded_data, 160,160,160)
inner_region, inner_mask, start, end = asm.inner_portion(enlarged_kspace=enlarged_cartesian, inner_sidelen=inner_wid)

In [7]:
cy = cx = cz = inner_wid // 2
r = 5  # radius in pixels; adjust to cover the gap
yy, xx, zz = np.ogrid[:inner_wid, :inner_wid, :inner_wid]
mask = (yy - cy)**2 + (xx - cx)**2 + (zz - cz)**2 > r**2 + 2

mask[(cy-6):(cy+7), cx, cz] = False   # line along axis 0
mask[cy, (cx-6):(cx+7), cz] = False   # line along axis 1
mask[cy, cx, (cz-6):(cz+7)] = False

mask2d = mask[:, :, cz] 

In [8]:
isolated_kspace = inner_region
innersidelen=inner_wid
sidelencart = 15
zte_radial_kspace = kspacelowres
sidelenrad = 10

In [9]:
def hankel(kspace, w):
    # kspace: (N_c, X, Y, Z)
    N_c = kspace.shape[0]
    N_x = kspace.shape[1] - w + 1
    N_y = kspace.shape[2] - w + 1
    N_z = kspace.shape[3] - w + 1
    data_matrix = []
    for c in range(N_c):
        onecoil_matrix = np.empty((w*w*w, N_x*N_y*N_z), dtype=np.complex64)
        col_index = 0
        for k in range(N_z):        # z outermost
            for i in range(N_y):
                for j in range(N_x):    # x innermost, matching your 2D ordering
                    mat = kspace[c, j:j+w, i:i+w, k:k+w]
                    onecoil_matrix[:, col_index] = mat.reshape(-1, order='F')
                    col_index += 1
        data_matrix.append(onecoil_matrix)
    return np.vstack(data_matrix), N_c, N_x, N_y, N_z

def hankel_H_averaged(data_matrix, n_coils, Nx, Ny, Nz, w):
    kspace = []
    split_arr = np.array(np.split(data_matrix, n_coils))

    n_win_x = Nx - w + 1
    n_win_y = Ny - w + 1
    n_win_z = Nz - w + 1

    for c in range(n_coils):
        singlecoil = split_arr[c]
        transposed = singlecoil.T          # one row per window

        recon = np.zeros((Nx, Ny, Nz), dtype=np.complex64)
        counts = np.zeros((Nx, Ny, Nz), dtype=np.float32)

        for index in range(len(transposed)):
            win = transposed[index].reshape((w, w, w), order='F')

            j = index % n_win_x
            i = (index // n_win_x) % n_win_y
            k = index // (n_win_x * n_win_y)

            recon[j:j+w, i:i+w, k:k+w] += win
            counts[j:j+w, i:i+w, k:k+w] += 1

        kspace.append(recon / counts)
    return np.array(kspace)

In [ ]:
import matplotlib

all_ztecoords= coorlowres.reshape(-1,3)
all_zteksp = kspacelowres.reshape(-1)

fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(30, 8))

grid_vol_zeropadding = np.abs(inner_region[0])
kx_centre_large = grid_vol_zeropadding.shape[0] // 2


im = ax[0].imshow(
    np.abs(grid_vol_zeropadding[kx_centre_large]),
    origin='lower',
    #norm=matplotlib.colors.LogNorm(),
)
fig.colorbar(im, ax=ax[0], label='kspace value')
ax[0].set_title('Cartesian')
ax[0].set_xlabel('kz')
ax[0].set_ylabel('ky')
ax[0].set_xlim(innersidelen/2 - sidelencart,innersidelen/2 + sidelencart)
ax[0].set_ylim(innersidelen/2 - sidelencart,innersidelen/2 + sidelencart)
ax[0].xaxis.set_minor_locator(plt.MultipleLocator(1, offset=0.5))
ax[0].yaxis.set_minor_locator(plt.MultipleLocator(1, offset=0.5))
ax[0].grid(visible=True, which='minor', linewidth=1)
ax[0].xaxis.set_major_locator(plt.MultipleLocator(1))
ax[0].yaxis.set_major_locator(plt.MultipleLocator(1))




yy, xx = np.mgrid[0:mask2d.shape[0], 0:mask2d.shape[1]]
im1 = ax[1].scatter(xx.ravel(), yy.ravel(), c=mask2d.ravel().astype(int),
                    cmap='viridis', s=300, marker='s', zorder=3)
fig.colorbar(im1, ax=ax[1], label='mask')
ax[1].set_title(f'Centre mask @ kz = {cz}')
ax[1].set_xlabel('x')
ax[1].set_ylabel('y')
ax[1].axis('equal')

c = mask2d.shape[0] / 2
ax[1].set_xlim(c - sidelencart, c + sidelencart)
ax[1].set_ylim(c - sidelencart, c + sidelencart)

ax[1].xaxis.set_minor_locator(plt.MultipleLocator(1, offset=0.5))
ax[1].yaxis.set_minor_locator(plt.MultipleLocator(1, offset=0.5))
ax[1].grid(visible=True, which='minor', linewidth=1)
ax[1].xaxis.set_major_locator(plt.MultipleLocator(1))
ax[1].yaxis.set_major_locator(plt.MultipleLocator(1))



tol = 0.5  # slab half-thickness; adjust to your coordinate units
mask = np.abs(all_ztecoords[:, 0]) < tol

im2 = ax[2].scatter(all_ztecoords[mask, 1], all_ztecoords[mask, 2],c=np.abs(kspacelowres[0]).reshape(-1)[mask],s=1)
fig.colorbar(im2, ax=ax[2], label='kspace value')
ax[2].set_title('Radial')
ax[2].set_xlabel('kx')
ax[2].set_ylabel('ky')
ax[2].axis('equal')
ax[2].xaxis.set_minor_locator(plt.MultipleLocator(1, offset=0.5))
ax[2].yaxis.set_minor_locator(plt.MultipleLocator(1, offset=0.5))
ax[2].grid(visible=True, which='minor', linewidth=1)
ax[2].xaxis.set_major_locator(plt.MultipleLocator(1))
ax[2].yaxis.set_major_locator(plt.MultipleLocator(1))
ax[2].set_xlim(-10,10)
ax[2].set_ylim(-10,10)

plt.show()

In [10]:
def softimpute_ALS_time(X_H, M_H, rank, lamda, n_iters):
    I = np.eye(rank)
    m, n = np.shape(X_H)
    U = np.random.randn(m, rank) + 1j * np.random.randn(m, rank)
    V = np.random.randn(n, rank) + 1j * np.random.randn(n, rank)

    D = I.copy()

    A = np.dot(U, D)
    B = np.dot(V, D)
    iter_count = 0
    ABt = A @ B.conj().T

    iter_times = []
    t_total_start = time.perf_counter()

    while iter_count < n_iters:
        t_start = time.perf_counter()

        X_star = np.where(M_H, X_H, ABt)
        X_star1H = X_star.copy()
        A = X_star @ B @ np.linalg.inv(B.conj().T @ B + lamda*I)
        ABt = A @ B.conj().T
        X_star = np.where(M_H, X_H, ABt)
        B = X_star.conj().T @ A @ np.linalg.inv(A.conj().T @ A + lamda*I)
        ABt = A @ B.conj().T
        iter_count += 1

        t_elapsed = time.perf_counter() - t_start
        iter_times.append(t_elapsed)

    t_total = time.perf_counter() - t_total_start
    t_mean = np.mean(iter_times)

    return (ABt, X_star1H, t_total, t_mean, iter_times)



def LORAKS_loop_timing(n_iters, window_size, zero_thresh, cartesian_inputkspace, zeropadded,dtg_mask, im_dim):
    ksp_forhankel = cartesian_inputkspace.copy()
    iter_count = 0
    deltas = []
    k_prev = None
    timings = {"hankel": [], "svd": [], "unlift": [], "consistency": [], "iter_total": [], "total": []}
    hankel_size = []
    t_loop_start = time.perf_counter()
    while iter_count < n_iters:
        t0 = time.perf_counter()

        hankel_matrix, n_coils, Numx, Numy = hankel(kspace=ksp_forhankel, w=window_size)
        t1 = time.perf_counter()

        U, S_reduced, Vh = asm.sig_val_thresholding_jax(data=hankel_matrix, zero_thresh=zero_thresh)
        data_recon = (U * S_reduced) @ Vh
        data_recon = jx.block_until_ready(data_recon)
        t2 = time.perf_counter()

        kspace_cart_coils_recon = hankel_H_averaged(
            data_recon, n_coils=ksp_forhankel.shape[0],
            Nx=ksp_forhankel.shape[1], Ny=ksp_forhankel.shape[2],
            w=window_size)
        t3 = time.perf_counter()

        kspace_cart_coils_consistent = np.where(dtg_mask, cartesian_inputkspace, kspace_cart_coils_recon)
        ksp_forhankel = np.asarray(kspace_cart_coils_consistent)  # ensure NumPy, on-host
        t4 = time.perf_counter()

        vec = ksp_forhankel[:, ~dtg_mask]
        if k_prev is not None:
            deltas.append(np.linalg.norm(vec - k_prev) / np.linalg.norm(k_prev))
        k_prev = vec.copy()

        timings["hankel"].append(t1 - t0)
        timings["svd"].append(t2 - t1)
        timings["unlift"].append(t3 - t2)
        timings["consistency"].append(t4 - t3)
        timings["iter_total"].append(t4 - t0)
        hankel_size.append(hankel_matrix.shape)

        iter_count += 1
    t_loop_end = time.perf_counter()
    timings["total"].append(t_loop_end - t_loop_start)
    output_kspace = ksp_forhankel.copy()
    filled_ksp = asm.rebuild(output_kspace=output_kspace, inner_mask=inner_mask,inner_start=start, inner_end=end,enlarged_kspace=zeropadded,resize_x=im_dim, resize_y=im_dim)
    im_grid0 = sp.ifft(filled_ksp, axes=(-2, -1))
    im_0 = np.sum(np.abs(im_grid0)**2, axis=0)**0.5

    return (im_0, filled_ksp, deltas, timings, hankel_size)

def LORAKS_imputeals(n_iters, window_size, cartesian_inputkspace, dtg_mask, zeropadded,rank, lamda, im_dim):
    ksp_forhankel = cartesian_inputkspace.copy()
    ksp_zerod = ksp_forhankel * dtg_mask
    hankel_matrix, n_coils, Numx, Numy = hankel(kspace=ksp_zerod, w=window_size)
    mask_coiled = np.broadcast_to(dtg_mask, (cartesian_inputkspace.shape))
    masked_hankel, *_ = hankel(kspace=mask_coiled, w=window_size)
    masked_hankel = np.real(masked_hankel) > 0.5
    
    
    filled_hankel, X_star1, t_total, t_mean, iter_times = softimpute_ALS_time(X_H = hankel_matrix, M_H = masked_hankel, rank = rank, lamda = lamda, n_iters=n_iters)

    kspace_cart_coils_recon = hankel_H_averaged(filled_hankel, n_coils=ksp_forhankel.shape[0], Nx=ksp_forhankel.shape[1], Ny=ksp_forhankel.shape[2], w=window_size)
    kspace_cart_coils_recon = np.where(dtg_mask, cartesian_inputkspace, kspace_cart_coils_recon)
    output_kspace = kspace_cart_coils_recon.copy()
    filled_ksp = asm.rebuild(output_kspace=output_kspace, inner_mask = inner_mask, inner_start = start, inner_end = end, enlarged_kspace=zeropadded, resize_x = im_dim, resize_y = im_dim)
    im_grid0 = sp.ifft(filled_ksp, axes=(-2, -1))
    im_0 = np.sum(np.abs(im_grid0)**2, axis=0)**0.5

    return (im_0, filled_ksp, masked_hankel, t_total, t_mean, iter_times)



def LORAKS_loop(n_iters, window_size, zero_thresh, cartesian_inputkspace, zeropadded,dtg_mask, im_dim):
    ksp_forhankel = cartesian_inputkspace.copy()
    iter_count = 0
    deltas = []
    k_prev = None
    while iter_count < n_iters:
        hankel_matrix, n_coils, Numx, Numy = hankel(kspace=ksp_forhankel, w=window_size)

        U, S_reduced, Vh = asm.sig_val_thresholding_jax(data=hankel_matrix, zero_thresh=zero_thresh)
        data_recon = (U * S_reduced) @ Vh

        kspace_cart_coils_recon = hankel_H_averaged(data_recon, n_coils=ksp_forhankel.shape[0], Nx=ksp_forhankel.shape[1], Ny=ksp_forhankel.shape[2], w=window_size)
        kspace_cart_coils_consistent = kspace_cart_coils_recon.copy()

        kspace_cart_coils_recon = np.where(dtg_mask, cartesian_inputkspace, kspace_cart_coils_recon)
        ksp_forhankel = kspace_cart_coils_consistent   

        vec = ksp_forhankel[:, ~dtg_mask]

        if k_prev is not None:
            deltas.append(np.linalg.norm(vec - k_prev) / np.linalg.norm(k_prev))
        k_prev = vec.copy()
        iter_count += 1

    output_kspace = ksp_forhankel.copy()
    filled_ksp = asm.rebuild(output_kspace=output_kspace, inner_mask = inner_mask, inner_start = start, inner_end = end, enlarged_kspace=zeropadded, resize_x = im_dim, resize_y = im_dim)
    im_grid0 = sp.ifft(filled_ksp, axes=(-2, -1))
    im_0 = np.sum(np.abs(im_grid0)**2, axis=0)**0.5
    
    return(im_0, filled_ksp, deltas)

In [ ]:
hankel_matrix, n_coils, Numx, Numy, Numz = hankel(kspace=inner_region, w=10)

In [ ]:
kspacerebuild = hankel_H_averaged(hankel_matrix, inner_region.shape[0], inner_region.shape[1], inner_region.shape[2], inner_region.shape[3], 10)